# Gradient Descent: Loss Hill Simulator

Gradient descent is an optimization algorithm. It improves a model by taking small steps in the direction that lowers the loss.

The idea goes back to nineteenth-century numerical optimization and became central to modern machine learning because training often means adjusting many parameters to reduce error. Neural networks, linear models, recommendation systems, and control systems all rely on this downhill-search pattern.

In this notebook, you will build it with small objects: data points, a line model, gradient steps, and a runner that learns a better line over time.

<details>
<summary>Big idea</summary>

Start with a guess. Measure how wrong it is. Use the gradient to figure out which way is downhill. Step a little. Repeat.

</details>

## 1. The Mental Model

Gradient descent needs a few moving parts:

- **Model**: a thing that makes predictions
- **Parameter**: a number the model can change, like slope or intercept
- **Loss**: how wrong the model is
- **Gradient**: the direction the loss increases fastest
- **Learning rate**: how big each step should be
- **Step**: update parameters in the opposite direction of the gradient

We will fit a line: `prediction = slope * x + intercept`.

<details>
<summary>Hint: why opposite the gradient?</summary>

The gradient points uphill. To lower the loss, move downhill: subtract a small amount of the gradient.

</details>

## 2. Build the Objects

Implementation plan:

1. `DataPoint` stores one training example.
2. `LineModel` stores slope and intercept.
3. `DescentStep` records each update.
4. `GradientDescentRunner` owns loss, gradients, and parameter updates.
5. `DescentReplay` prints the learning path.

<details>
<summary>Implementation hint</summary>

For mean squared error, the gradients are averages of the prediction errors. The slope gradient cares about `error * x`; the intercept gradient cares about `error`.

</details>

**Object model.** Define `DataPoint`, `LineModel`, the named objects used by the next examples.


In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class DataPoint:
    x: float
    y: float

    def __str__(self) -> str:
        return f"({self.x:g}, {self.y:g})"

@dataclass(frozen=True)
class LineModel:
    slope: float
    intercept: float

    def predict(self, x: float) -> float:
        return self.slope * x + self.intercept

    def __str__(self) -> str:
        return f"y = {self.slope:.3f}x + {self.intercept:.3f}"


**Trace model.** Define `DescentStep`, `DescentResult`, the structure used to capture replayable algorithm state.


In [ ]:
@dataclass
class DescentStep:
    round_number: int
    model: LineModel
    loss: float
    slope_gradient: float
    intercept_gradient: float
    slope_change: float
    intercept_change: float
    note: str

@dataclass
class DescentResult:
    model: LineModel
    steps: list[DescentStep]


**Algorithm engine.** Define `GradientDescentRunner`, the class that runs the main simulation or algorithm.


In [ ]:
class GradientDescentRunner:
    def __init__(self, data: list[DataPoint], starting_model: LineModel, learning_rate: float):
        if not data:
            raise ValueError("Gradient descent needs at least one data point.")
        if learning_rate <= 0:
            raise ValueError("Learning rate must be positive.")

        self.data = data
        self.starting_model = starting_model
        self.learning_rate = learning_rate

    def loss(self, model: LineModel) -> float:
        squared_errors = [(model.predict(point.x) - point.y) ** 2 for point in self.data]
        return sum(squared_errors) / len(self.data)

    def gradients(self, model: LineModel) -> tuple[float, float]:
        slope_total = 0.0
        intercept_total = 0.0

        for point in self.data:
            error = model.predict(point.x) - point.y
            slope_total += error * point.x
            intercept_total += error

        scale = 2 / len(self.data)
        return scale * slope_total, scale * intercept_total

    def run(self, rounds: int = 30, tolerance: float = 0.0001) -> DescentResult:
        model = self.starting_model
        steps = [
            DescentStep(
                round_number=0,
                model=model,
                loss=self.loss(model),
                slope_gradient=0.0,
                intercept_gradient=0.0,
                slope_change=0.0,
                intercept_change=0.0,
                note="Starting guess.",
            )
        ]

        for round_number in range(1, rounds + 1):
            slope_gradient, intercept_gradient = self.gradients(model)
            slope_change = -self.learning_rate * slope_gradient
            intercept_change = -self.learning_rate * intercept_gradient
            model = LineModel(
                slope=model.slope + slope_change,
                intercept=model.intercept + intercept_change,
            )
            steps.append(
                DescentStep(
                    round_number=round_number,
                    model=model,
                    loss=self.loss(model),
                    slope_gradient=slope_gradient,
                    intercept_gradient=intercept_gradient,
                    slope_change=slope_change,
                    intercept_change=intercept_change,
                    note=f"Move downhill with learning rate {self.learning_rate}.",
                )
            )

            if abs(slope_change) + abs(intercept_change) <= tolerance:
                break

        return DescentResult(model=model, steps=steps)


## 3. Create a Tiny Training Set

Our model will learn a line from noisy points. The hidden pattern is close to `y = 2x + 1`, but the algorithm only sees examples.

<details>
<summary>Why use noisy points?</summary>

Real data is rarely perfect. Gradient descent should find a line that is good overall, not a line that magically hits every point.

</details>

In [2]:
training_points = [
    DataPoint(-3, -5.1),
    DataPoint(-2, -3.0),
    DataPoint(-1, -1.0),
    DataPoint(0, 1.2),
    DataPoint(1, 3.1),
    DataPoint(2, 4.9),
    DataPoint(3, 7.2),
]

starting_model = LineModel(slope=0.0, intercept=0.0)
learning_rate = 0.05

print("Training points:")
for point in training_points:
    print(f"  {point}")

print(f"\nStarting model: {starting_model}")

Training points:
  (-3, -5.1)
  (-2, -3)
  (-1, -1)
  (0, 1.2)
  (1, 3.1)
  (2, 4.9)
  (3, 7.2)

Starting model: y = 0.000x + 0.000


## 4. Run Gradient Descent

Each round:

1. Predict with the current line.
2. Measure mean squared error.
3. Compute gradients for slope and intercept.
4. Update the line by subtracting a learning-rate-sized gradient step.

<details>
<summary>Formula hint</summary>

If `error = prediction - actual`, then:

- slope gradient = average of `2 * error * x`
- intercept gradient = average of `2 * error`

</details>

In [3]:
runner = GradientDescentRunner(
    data=training_points,
    starting_model=starting_model,
    learning_rate=learning_rate,
)
result = runner.run(rounds=40)

first_step = result.steps[0]
last_step = result.steps[-1]

print(f"Initial loss: {first_step.loss:.4f}")
print(f"Final loss:   {last_step.loss:.4f}")
print(f"Learned line: {result.model}")

print("\nPredictions:")
for point in training_points:
    prediction = result.model.predict(point.x)
    print(f"  x={point.x:>4g} actual={point.y:>5.1f} predicted={prediction:>5.2f}")

Initial loss: 17.5586
Final loss:   0.0108
Learned line: y = 2.029x + 1.027

Predictions:
  x=  -3 actual= -5.1 predicted=-5.06
  x=  -2 actual= -3.0 predicted=-3.03
  x=  -1 actual= -1.0 predicted=-1.00
  x=   0 actual=  1.2 predicted= 1.03
  x=   1 actual=  3.1 predicted= 3.06
  x=   2 actual=  4.9 predicted= 5.08
  x=   3 actual=  7.2 predicted= 7.11


## 5. Replay the Descent

A replay makes the algorithm feel less mysterious. Each snapshot shows the current line, the gradients, the parameter changes, and the new loss.

<details>
<summary>Reading the replay</summary>

Large gradients mean the model is far from a good answer. As the model improves, the loss and update sizes usually shrink.

</details>

**Trace model.** Define `DescentReplay`, the structure used to capture replayable algorithm state.


In [ ]:
class DescentReplay:
    def __init__(self, result: DescentResult):
        self.result = result

    def show(self, first_rounds: int = 8) -> None:
        visible_steps = self.result.steps[: first_rounds + 1]
        final_step = self.result.steps[-1]

        if final_step.round_number > first_rounds:
            visible_steps.append(final_step)

        for index, step in enumerate(visible_steps):
            if index == first_rounds + 1 and step.round_number != first_rounds + 1:
                print("  ...")

            print(f"Round {step.round_number:>2}: loss={step.loss:>7.4f} | {step.model}")

            if step.round_number > 0:
                print(
                    f"          gradients=({step.slope_gradient:>7.3f}, {step.intercept_gradient:>7.3f}) "
                    f"changes=({step.slope_change:>7.3f}, {step.intercept_change:>7.3f})"
                )


**Inspect the result.** Evaluate the expression and read the output before changing parameters.


In [ ]:
DescentReplay(result).show(first_rounds=7)


## 6. Experiments

Gradient descent is sensitive to the learning rate. Try a step size that is tiny, useful, aggressive, and too large.

<details>
<summary>Experiment hint</summary>

If the learning rate is too small, learning is slow. If it is too large, the model may jump across the valley and make the loss worse.

</details>

In [5]:
def compare_learning_rate(rate: float, rounds: int = 12) -> None:
    experiment_runner = GradientDescentRunner(
        data=training_points,
        starting_model=starting_model,
        learning_rate=rate,
    )
    experiment_result = experiment_runner.run(rounds=rounds, tolerance=0.0)
    start_loss = experiment_result.steps[0].loss
    end_loss = experiment_result.steps[-1].loss
    direction = "improved" if end_loss < start_loss else "got worse"

    print(
        f"rate={rate:<5g} rounds={len(experiment_result.steps) - 1:>2} "
        f"start_loss={start_loss:>9.3f} end_loss={end_loss:>9.3f} "
        f"-> {direction:>9} | {experiment_result.model}"
    )


for rate in [0.005, 0.05, 0.2, 0.3]:
    compare_learning_rate(rate)

rate=0.005 rounds=12 start_loss=   17.559 end_loss=    7.045 ->  improved | y = 0.786x + 0.118
rate=0.05  rounds=12 start_loss=   17.559 end_loss=    0.097 ->  improved | y = 2.024x + 0.748
rate=0.2   rounds=12 start_loss=   17.559 end_loss=    0.011 ->  improved | y = 2.024x + 1.041
rate=0.3   rounds=12 start_loss=   17.559 end_loss=52907.050 -> got worse | y = -112.979x + 1.043


## Visual Trace + Rigor Studio

**Problem frame.** Optimize a model by following the local slope of a loss function.

**Interactive animation target.** Animate the current point moving on a loss curve or surface with learning-rate control.

**Correctness handle.** For a sufficiently small step on a smooth function, moving against the gradient locally lowers loss.

**Complexity handle.** O(iterations * cost_of_gradient), with convergence depending on curvature and step size.

**Failure mode to test.** A large learning rate can overshoot or diverge; nonconvex losses can trap local minima.

**Studio task.** Run three learning rates and classify each as slow, stable, or unstable.


In [ ]:
from pathlib import Path
import sys
from IPython.display import display

for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "courseware").exists():
        sys.path.insert(0, str(candidate))
        break

from courseware.visual_tracing import AlgorithmPlayer, gradient_descent_trace, render_gradient_descent, render_trace_table

f = lambda x: (x - 1.5) ** 2 + 0.4
df = lambda x: 2 * (x - 1.5)
trace = gradient_descent_trace(f, df, x0=-3, learning_rate=0.25, steps=10, domain=(-4, 4))
display(render_trace_table(trace))
AlgorithmPlayer(trace, render_gradient_descent).display()
